# PlanktoScan AI - Colab backup
Runs the same scripts as the local setup. Backup only: the main results were produced locally on an RTX 3050.

1. Runtime -> Change runtime type -> GPU.
2. Replace `REPO_URL` below with your (private) repository URL. Private repos need a GitHub token.
3. The ZooLake download (492 MB, SHA-256 checked) runs from the official Eawag source.

In [ ]:
REPO_URL = "https://github.com/<your-user>/planktoscan-ai.git"  # <- change me
!git clone $REPO_URL planktoscan-ai
%cd planktoscan-ai

In [ ]:
# Colab already ships CUDA PyTorch; install the rest
!pip install -q -r requirements.txt
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
!python -m scripts.download_zoolake
!python -m scripts.inventory_zoolake
!python -m scripts.make_splits

In [ ]:
# Phase 3/4: classifier ablations (val macro F1 decides; see README)
!python -m scripts.train_classifier --name abl_pre0 --use-preprocess 0 --weighting none --epochs 25
!python -m scripts.train_classifier --name abl_pre1 --use-preprocess 1 --weighting none --epochs 25
!python -m scripts.train_classifier --name cls_w0 --weighting none
!python -m scripts.train_classifier --name cls_w1 --weighting inverse_sqrt_freq
!python -m scripts.select_classifier

In [ ]:
# Phase 5: synthetic frames + detector
!python -m scripts.make_synthetic
!python -m scripts.train_detector
!python -m scripts.tune_detector
!python -m scripts.choose_threshold

In [ ]:
# Phase 8: tests + full evaluation
!python -m pytest -q
!python -m scripts.evaluate
print(open('results/final_summary.md').read())